**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/payne-2020"], check=True)
    os.chdir("/content/encore/entries/payne-2020")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Payne et al. (2020) on SPARC:206

Paper. Payne, Wiedmann, Eiber, Wong, Senn, Osborne, Keast & Fallon (2020), "Recording of
Electrically Evoked Neural Activity and Bladder Pressure Responses in Awake Rats Chronically
Implanted With a Pelvic Nerve Array", Front. Neurosci. 14, 619275, doi:10.3389/fnins.2020.619275.
The claims file (`claims/payne-2020.yaml`) was written from the version of record (Europe PMC
PMC7773906): main text, Methods, Table 1, figure legends and Discussion; the paper has no
supplement and links no code.

Dataset. SPARC:206, "Recording of electrically evoked neural activity and bladder pressure
responses in awake rats chronically implanted with a pelvic nerve array", version 1,
DOI 10.26275/kkmb-vun5, read from the public bucket `sparc-prod-aod-discover-publish50-use1` at
the object versions listed by that version's manifest (manifest object version
`Ag7Kwl87aKqHqVTtho8dzikC2BfNyghN`). Version 1 is pinned because it is the only published
version of the dataset (so also the most recent) and the paper cites no dataset version; its
metadata name this paper as the article that describes it.

Claims.

| Claim | Status |
|---|---|
| anes-ecap-recorded | tested |
| anes-ecap-latencies | tested |
| anes-conduction-velocity | tested |
| anes-threshold-order | tested |
| fig2c-p3-above-p1 | tested |
| fig2ab-one-rat-p2-below-p1 | tested |
| awake-ecap-latencies | tested |
| awake-threshold-increase | blocked, metadata_insufficient (no link between the week 0 and week 1 recordings) |
| impedance-increase | tested |
| electrode-failures | blocked, data_missing (per-electrode impedances for 8 of 10 arrays) |
| colonic-pressure | tested |
| no-fecal-expulsion-or-erection | blocked, data_missing |
| table1-minimum-pressure | tested |
| table1-threshold-pressure | tested |
| table1-peak-pressure | tested |
| table1-contraction-duration | tested |
| table1-intervoid-interval | tested |
| table1-hfpo-frequency | tested |
| table1-comparable-to-published | tested (with_inputs) |
| stim-evoked-voiding | blocked, data_missing (no urine-release annotation; stimulation in 4 rats) |
| stim-void-trials | blocked, data_missing |
| stim-early-void | blocked, data_missing |
| stim-nonvoiding | blocked, data_missing |
| stim-voiding-to-nonvoiding | blocked, data_missing |
| fig3e-eight-week-responses | blocked, data_missing (no stimulation in the 8-week recordings) |
| fig4-atf3-rare | blocked, data_missing |
| fig4ef-axons-intact | blocked, data_missing |
| fig5-cfos-distribution | blocked, data_missing |
| fig5-cfos-no-side-difference | blocked, data_missing |
| fig5-no-lamina1 | blocked, data_missing |
| macroscopic-no-pathology | blocked, data_missing |

In [ ]:
import time, resource
T0 = time.time()
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import pelvic as P
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

man = P.manifest()
print("files in the version-1 manifest:", len(man))
used = sorted([p for p in man if p.startswith("files/") and not p.endswith(".pxp") and
               not p.endswith(".py")])
print(pd.DataFrame({"path": used, "size": [man[p]["size"] for p in used],
                    "sha256": [P.sha256_hex(p) for p in used]}).to_string())

## Data check

The dataset follows the SPARC Data Structure. The data check reads the top-level sheets, the
folder manifests, and the attributes of every recording, and compares them with what the paper
describes.

In [ ]:
dd = P.read_excel("files/dataset_description.xlsx", header=None)
for _, r in dd.iterrows():
    v = [str(x) for x in r.values[3:] if str(x) != "nan"]
    print(f"{str(r[0])[:32]:32s} | {' | '.join(v)[:150]}")
sj = P.read_excel("files/subjects.xlsx")
print(sj.columns.tolist())
print(sj[["subject_id", "experimental group", "age", "sex", "Implant Side", "Recording Day (post-surgery)"]].to_string())
print("\nall columns, including pool_id and the experimental log file name:")
print(sj.drop(columns=["species", "strain", "RRID for strain", "protocol title", "protocol.io location"]).to_string())
print("species, strain, RRID, protocol:", {c: sj[c].dropna().unique().tolist() for c in ["species", "strain", "RRID for strain", "protocol title", "protocol.io location"]})
print("\nsubjects per group:\n", sj["experimental group"].value_counts().to_string())
print("total subjects:", sj.subject_id.nunique(), "(the paper reports 24 rats in all)")

In [ ]:
for folder in ["primary", "derivative", "code", "docs", "protocol"]:
    m = P.read_excel(f"files/{folder}/manifest.xlsx")
    print(folder, len(m), "rows; columns", m.columns.tolist())
    if folder != "primary":
        print(m.to_string())
readme = P.read_bytes("files/README.txt").decode()
print([l for l in readme.splitlines() if "c-Fos" in l])
print("files whose name mentions fos, atf, histolog or image:",
      [p for p in man if any(k in p.lower() for k in ["fos", "atf", "histo", "tif", "jpg", "czi"])])

### ECAP recordings

Five acute rats (urethane, `acute ECAP`) and seven chronic recordings (`chronic ECAP`, recording
day 0 or 7). Every trace's attributes are read and checked against the Methods (100 kHz
sampling, 300 to 5000 Hz filter, stimulation on E1-E2 and recording on E3-E4, currents 0 to 2 mA
in 0.1 mA steps, two sets of 50 averages).

In [ ]:
E = {}
rows = []
for s in P.ACUTE + P.CHRONIC:
    tab, X, t, a = P.load_ecap(s)
    E[s] = (tab, X, t, a)
    L = P.ecap_levels(tab, X)
    day = int(sj.set_index("subject_id").loc["sub-" + s, "Recording Day (post-surgery)"])
    rows.append(dict(sub=s, group="acute" if s in P.ACUTE else "chronic", day=day,
                     traces=len(tab), currents=len(L), pair=tab.pair.iloc[0],
                     rec=a["SourceChannel"].decode(), fs_kHz=a["SampleRate"] / 1000,
                     filt=f'{a["FilterLowCorner"]:.0f}-{a["FilterHighCorner"]:.0f} {a["FilterUnits"].decode()}',
                     averages=a["Averages"], t0_ms=t[0], t1_ms=round(t[-1], 3),
                     units=a["WaveUnits"].decode(),
                     min_mA=round(L[0][0], 4), max_mA=round(L[-1][0], 3),
                     noise_uV=round(P.noise_sd(L, t), 3),
                     trials_gt2=int((tab.trial > 2).sum())))
ecap_meta = pd.DataFrame(rows)
print(ecap_meta.to_string())
for s in ["12-005", "12-051"]:
    print(s, "currents (mA):", np.round([l[0] for l in P.ecap_levels(*E[s][:2])], 3))

In [ ]:
# Every attribute of every trace and of /Stimuli, per file (values that differ between
# traces of a file are listed together), and the deposited schema of the ECAP files.
for s in P.ACUTE + P.CHRONIC:
    with P.h5(f"files/primary/sub-{s}/sub-{s}_ECAP.hdf5") as f:
        att = {}
        for n in f["ECAPs"]:
            for k, v in f["ECAPs"][n].attrs.items():
                att.setdefault(k, set()).add(str(np.asarray(v).ravel()[0]))
        st = {k: str(np.asarray(v).ravel()[0]) for k, v in f["Stimuli"].attrs.items()}
        root = dict(f.attrs)
    print(s, {k: sorted(v) for k, v in att.items()}, "| Stimuli:", st, "| root attrs:", root)
X_all = np.concatenate([E[s][1].ravel() for s in E])
print("largest |value| over all traces: %.4f mV (%.1f uV)" % (np.abs(X_all).max() / 1000, np.abs(X_all).max()))
print(P.read_bytes("files/docs/hdf5_ECAP_schema.json").decode())

No trace or file attribute names a gain, and the schema gives none; the Methods give a
voltage gain of 10^2. Whether the stored mV are before or after that gain cannot be told from
the files. The schema gives `TimeUnits` "s" for a `StartTime` of 0.6 and a `Duration` of 8, while
the files say "ms"; it does not say where time zero is. Both are open: the time origin is run
under the claims file's two readings, and a gain of 100 in the stored values is run as a
variant of the threshold (a pass level of 5 uV in the stored units).

In [ ]:
# Recordings deposited twice: compare every trace with every other across files.
import hashlib
h = {}
for s, (tab, X, t, a) in E.items():
    for i in range(len(tab)):
        h.setdefault(hashlib.md5(np.round(X[i], 9).tobytes()).hexdigest(), []).append((s, tab.name[i]))
dups = [v for v in h.values() if len(v) > 1]
print("ECAP traces:", sum(len(v) for v in h.values()), "identical groups:", len(dups))
for d in dups[:10]:
    print(d)
# whole-file similarity: correlation of the mean trace at the highest current between files
tw = (E["12-005"][2] >= 1.2) & (E["12-005"][2] <= 7.0)
M = {s: (lambda L: (L[-1][1] + L[-1][2]) / 2)(P.ecap_levels(*E[s][:2]))[tw] for s in E}
C = pd.DataFrame([[np.corrcoef(M[a], M[b])[0, 1] for b in E] for a in E], index=list(E), columns=list(E))
print("correlation of the mean trace at the highest current, 1.2 to 7 ms, between files:")
print(C.round(2).to_string())

Clock check for the ECAPs. Each trace is aligned to its stimulus pulse (it starts 0.6 ms
after the pulse onset, by its `StartTime` attribute). If the responses are time-locked to the
pulse, the response window (1.2 to 6 ms) grows with current while a late window (7 to 8.6 ms)
does not. Spearman correlation of the RMS in each window with current, per file.

In [ ]:
rows = []
for s, (tab, X, t, a) in E.items():
    L = P.ecap_levels(tab, X)
    cur = [l[0] for l in L]
    def rms(x, lo, hi):
        y = x - P.baseline(x, t); return float(np.sqrt(np.mean(y[(t >= lo) & (t <= hi)] ** 2)))
    early = [rms((r1 + r2) / 2, 1.2, 6.0) for _, r1, r2 in L]
    late = [rms((r1 + r2) / 2, 7.0, 8.6) for _, r1, r2 in L]
    rows.append(dict(sub=s, rho_response=stats.spearmanr(cur, early)[0],
                     p_response=stats.spearmanr(cur, early)[1],
                     rho_late=stats.spearmanr(cur, late)[0],
                     rms_resp_top_uV=early[-1], rms_resp_lowest_uV=early[0]))
print(pd.DataFrame(rows).round(4).to_string())

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(18, 13))
for ax, s in zip(axs.ravel(), E):
    tab, X, t, a = E[s]
    L = P.ecap_levels(tab, X)
    sc = np.percentile(np.abs(X[:, t > 1.2]), 99)
    for k, (mA, r1, r2) in enumerate(L):
        ax.plot(t, (r1 + r2) / 2 / sc * 0.5 + k, lw=0.6)
    ax.set_ylim(-1, len(L) + 1); ax.set_xlim(0.6, 8.6)
    ax.set_title(f"{s}: one row per current, {L[0][0]:.3f} to {L[-1][0]:.2f} mA", fontsize=9)
    ax.set_xlabel("ms from pulse onset")
plt.tight_layout(); plt.show()
print("Each trace is scaled by the 99th percentile of |V| after 1.2 ms of its file; traces "
      "within 0.6 to 1.2 ms (artefact) run off the rows.")

### Cystometry recordings

Each `*_cystometry.hdf5` holds bladder pressure (`/Cystometry`), the voids its authors marked
(`/Voids`) and, in four rats, the stimulation trains (`/Stimuli`). The derivative table
`cystometry-dataset.xlsx` holds the Table 1 parameters for three cycles per rat.

In [ ]:
CY = {}
rows = []
for s in P.CYSTO:
    W, V, S = P.load_cysto(s)
    CY[s] = (W, V, S)
    r = sj.set_index("subject_id").loc["sub-" + s]
    rows.append(dict(sub=s, group=r["experimental group"], day=r["Recording Day (post-surgery)"],
                     waves=len(W), fs=sorted({w["attrs"]["SampleRate"] for w in W}),
                     units=sorted({w["attrs"]["WaveUnits"].decode() for w in W}),
                     channel=sorted({w["attrs"]["SourceChannel"].decode() for w in W}),
                     minutes=round(sum(w["t"][-1] - w["t"][0] for w in W) / 60, 1),
                     p_min=round(min(np.nanmin(w["p"]) for w in W), 1),
                     p_max=round(max(np.nanmax(w["p"]) for w in W), 1),
                     voids=len(V), stim_trains=0 if S is None else len(S),
                     stim_mA=None if S is None else sorted(set(np.round(S["Stim"], 2)) if "Stim" in S else {S.attrs.get("Stim")}),
                     offset=sorted({w["attrs"].get("AppliedOffset", np.nan) for w in W})))
cy_meta = pd.DataFrame(rows)
print(cy_meta.to_string())
print("rats with stimulation trains:", list(cy_meta[cy_meta.stim_trains > 0]["sub"]))

In [ ]:
dv = P.read_excel("files/derivative/cystometry-dataset.xlsx")
print(dv.columns.tolist(), len(dv))
print(dv.groupby("subject_id").agg(file=("file", "first"), cycles=("cycle", "nunique"),
                                   rows=("value", "size"), missing=("value", lambda x: int(x.isna().sum()))).to_string())
print(dv.groupby(["parameter_id", "parameter"]).value.agg(["count", "min", "max"]).round(2).to_string())
# relative pressures are absolute minus minimum, as the Methods say
w = dv.pivot_table(index=["subject_id", "cycle"], columns="parameter_id", values="value")
print("max |init - eov - init_rel|:", float(np.nanmax(np.abs(w.mmhg_init - w.mmhg_eov - w.mmhg_init_rel))),
      " max |peak - eov - peak_rel|:", float(np.nanmax(np.abs(w.mmhg_peak - w.mmhg_eov - w.mmhg_peak_rel))))

The derivative names its rats `R18-208` and `12_081`; the primary folders are `sub-18-208` and
`sub-12-081`. The match is by number. The derivative's values are compared with the primary
`/Voids` of the same rats below, to see whether the derivative was computed from the deposited
recordings.

In [ ]:
def to_sub(x):
    return x.replace("R", "").replace("_", "-")
rows = []
for sid, g in dv.groupby("subject_id"):
    s = to_sub(sid)
    V = CY[s][1]
    d = g.pivot_table(index="cycle", columns="parameter_id", values="value")
    for c in d.index:
        v = V.iloc[c - 1]
        rows.append(dict(sub=s, cycle=c, deriv_peak=d.loc[c, "mmhg_peak"], voids_peak=v.PeakPressure,
                         deriv_thr=d.loc[c, "mmhg_init"], voids_thr=v.ThresholdPressure,
                         deriv_dur=d.loc[c, "contraction_dur_s"], voids_dur=v.ContractionDuration,
                         deriv_ivi_min=d.loc[c, "ivi_minute"],
                         voids_ivi_min=v.VoidInterval / 60 if "VoidInterval" in V else np.nan))
cmp = pd.DataFrame(rows)
print(cmp.round(2).to_string())
print("cycles where the derivative peak pressure equals the /Voids peak within 0.5 mmHg:",
      int((np.abs(cmp.deriv_peak - cmp.voids_peak) < 0.5).sum()), "of", len(cmp))

Clock check for the cystometry stimulation. In the four rats with stimulation trains, the
bladder pressure change over the 60 s after each train onset (mean of the 60 s after minus the
mean of the 30 s before) is compared with the same measure at times shifted by 150 s, 300 s
and 450 s, which do not follow a train.

In [ ]:
def pchange(W, t0, pre=30, post=60):
    for w in W:
        t, p = w["t"], w["p"]
        if t[0] <= t0 - pre and t0 + post <= t[-1]:
            return float(np.nanmean(p[(t >= t0) & (t < t0 + post)]) - np.nanmean(p[(t >= t0 - pre) & (t < t0)]))
    return np.nan
rows = []
for s in cy_meta[cy_meta.stim_trains > 0]["sub"]:
    W, V, S = CY[s]
    times = S["Time"].values if "Time" in S else np.array([S.attrs["Time"]])
    for t0 in times:
        rows.append(dict(sub=s, onset=t0, change_at_train=pchange(W, t0),
                         change_shifted=np.nanmean([pchange(W, t0 + d) for d in (150, 300, 450)]) if np.isfinite([pchange(W, t0 + d) for d in (150, 300, 450)]).any() else np.nan))
cc = pd.DataFrame(rows)
print(cc.round(2).to_string())
print("median change after trains %.2f mmHg, at shifted times %.2f mmHg" %
      (cc.change_at_train.median(), cc.change_shifted.median()))

### Colonic pressure recordings

Three urethane-anaesthetised rats (`colon pressure recording`), colonic balloon pressure with
the stimulation trains. The Methods give 10 Hz, 100 us pulses at 1 mA.

In [ ]:
COL = {}
rows = []
for s in P.COLON:
    with P.h5(f"files/primary/sub-{s}/sub-{s}_recording.hdf5") as f:
        d = f["ColonPressure"]["Wave01"]
        a = {k: np.asarray(v).ravel()[0] for k, v in d.attrs.items()}
        p = d[()].ravel()
        tt = float(a["StartTime"]) + np.arange(p.size) / float(a["SampleRate"])
        St = f["Stimuli"]["Time"][()].ravel()
        sa = {k: np.asarray(v).ravel()[0] for k, v in f["Stimuli"].attrs.items()}
    csv = P.read_csv(f"files/primary/sub-{s}/sub-{s}_stimuli.csv")
    assert np.allclose(csv.time_s.values, St)
    COL[s] = (tt, p, St, sa)
    rows.append(dict(sub=s, fs=a["SampleRate"], units=a["WaveUnits"].decode(),
                     channel=a["SourceChannel"].decode(), seconds=round(tt[-1], 1),
                     p_range=(round(np.nanmin(p), 1), round(np.nanmax(p), 1)), nan_samples=int(np.isnan(p).sum()), stim_entries=len(St),
                     min_gap_s=round(np.diff(St).min(), 2), mA=sa["Stim"], pw_ms=sa["StimPw"],
                     rate_Hz=sa["StimRate"], train_s=round(sa["StimDuration"], 2),
                     pulses=sa["StimCount"]))
print(pd.DataFrame(rows).to_string())

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15, 9))
for ax, s in zip(axs, P.COLON):
    tt, p, St, sa = COL[s]
    ax.plot(tt, p, lw=0.5)
    for t0 in St:
        ax.axvspan(t0, t0 + sa["StimDuration"], color="r", alpha=0.2)
    ax.set_title(f"{s}: colonic pressure (mmHg), trains shaded; range {np.nanmin(p):.1f} to {np.nanmax(p):.1f}")
plt.tight_layout(); plt.show()

Clock check for the colonic recordings. Mean pressure in 2 s bins from 30 s before to 40 s
after each recorded stimulus time, relative to the mean 30 to 40 s before it.

In [ ]:
lags = np.arange(-30, 40, 2.0)
ETA = {}
for s in P.COLON:
    tt, p, St, sa = COL[s]
    ETA[s] = [np.nanmean([np.nanmean(p[(tt >= t0 + l) & (tt < t0 + l + 2)]) - np.nanmean(p[(tt >= t0 - 40) & (tt < t0 - 30)])
                          for t0 in St if t0 - 40 >= tt[0] and t0 + 40 <= tt[-1]]) for l in lags]
print(pd.DataFrame(ETA, index=[f"{l:+.0f} s" for l in lags]).round(2).T.to_string())

In 15-758 the pressure rises by about 0.65 mmHg in the 10 s before each recorded stimulus time
and returns to baseline at that time; the recorded times are 10 s apart from the response, the
length of one train. Either the times mark the end of each train, or they are offset by 10 s.
The files do not say which. In 12-019 and 12-020 the averages drift by several mmHg over tens of
seconds, and no time-locked step stands out.

### Impedances

Per-electrode impedances in `sub-18-*_impedances.csv` (eight rats) and the per-array values of
the derivative `chronic-impedances.pzfx` (a GraphPad Prism file).

In [ ]:
imp = []
for s in [x for x in P.CYSTO if f"files/primary/sub-{x}/sub-{x}_impedances.csv" in man]:
    d = P.read_csv(f"files/primary/sub-{s}/sub-{s}_impedances.csv")
    imp.append(d)
imp = pd.concat(imp, ignore_index=True)
imp = imp.rename(columns={"Z_e4": "Z_E4"})
print(imp.drop(columns=["subject"]).to_string())
Z = ["Z_E1", "Z_E2", "Z_E3", "Z_E4"]
print("value range (Ohm):", imp[Z].min().min(), "to", imp[Z].max().max(), "; arrays:", imp.ANIMAL.nunique())
pz = P.read_pzfx("files/derivative/chronic-impedances.pzfx")
print({k: v for k, v in pz.items()})
print("per-array columns, non-empty values:", {k: int(np.isfinite(v).sum()) for k, v in pz.items()})

In [ ]:
print("subjects in subjects.xlsx not matched by any primary file:",
      [x for x in sj.subject_id if not any(p.startswith("files/primary/" + x + "/") for p in man)])
print("peak memory so far (GB): %.2f" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

### What the data check finds

- The ECAP files are sampled at 200 kHz and carry a 500 to 3000 Hz 4th-order Butterworth
  filter, where the Methods state 100 kHz and 300 to 5000 Hz. The acute files step the current
  on a logarithmic scale (`CL` 80 to 250, about 0.074 to 1.60 mA, plus `CL0` at 0.0175 mA),
  not 0 to 2 mA in 0.1 mA steps; four of the chronic files (12-050 to 12-053) do use 0.1 mA
  steps, to 1 or 2 mA. Stimulation is on E3-E4 with recording on E1-E2 in some files
  (12-008, 12-050, 12-053, 12-056), and in the reverse polarity (E2 to E1) in 12-006 and 12-007,
  where the Methods give E1-E2 for stimulation.
- The noise SD (lowest current, after 1 ms) is 0.02 to 0.06 uV in four files and 0.2 to 0.7
  uV in the others, so the paper's 0.05 uV threshold lies within three noise SDs in every file.
- The seven chronic ECAP files are seven subject identifiers, four at day 0 and three at day 7.
  Nothing links a day-7 file to a day-0 file, so no rat can be followed from week 0 to week 1.
  The paper has three (Methods) or four (Results) awake rats.
- The dataset lists 29 subjects; the paper reports 24 rats. `pool_id` is empty for every
  subject, and the experimental log file name is given only for the 18-* cystometry rats, so
  nothing in subjects.xlsx links the day-0 and day-7 awake ECAP files.
- No ECAP attribute or schema entry gives the amplifier gain, and the ECAP schema gives
  `TimeUnits` "s" where the files have "ms"; it does not define time zero. The cystometry
  schema lists the `/Voids` columns with units only, without saying whether a void means urine
  release.
- Stimulation trains are in four cystometry files (18-196, 18-243, 18-262, 18-294; 1 to 5
  trains each); no file marks whether urine was released. The two 8-week rats (15-759, 15-760)
  have no stimulation. The README promises summary c-Fos data, which no file holds; there are
  no histology images or observation notes.
- The colonic recordings were stimulated at 1.114 mA, 200 us, 10 Hz, 4 or 5 s (12-019,
  12-020) and at 0.541 mA, 25 us, 15 Hz, 10 s (15-758), where the Methods give 1 mA, 100 us,
  10 Hz (and the claims, 10 s trains).

The numbers behind each point are printed above.

## Anaesthetised ECAPs

All six anaesthetised-ECAP claims use the same peak identification (an assumption: see the
helper module's docstring). Candidates are the positive local maxima after the artefact end in
the mean of the two recordings at the highest current, with prominence at least k noise SD
(k = 2, 3, 5) and amplitude at least 0.05 uV. Four rules assign P1 to P3: `first`, the first
three candidates in time (the main rule); `largest`, the three largest, in time order;
`window`, the triple inside the paper's P1, P2 and P3 ranges (each widened by 10%) with the
largest summed amplitude; `inv`, the first rule on the inverted traces. Each peak is then
followed to lower currents as the local maximum nearest its latency within +-0.3 ms. The main
reading is rule `first`, artefact end (a), k = 3.

In [ ]:
A = {}
for rule in P.RULES:
    for art in "ab":
        for k in (2.0, 3.0, 5.0):
            for s in P.ACUTE:
                A[(rule, art, k, s)] = P.analyse_ecap(s, art=art, k_noise=k, rule=rule, windows=P.WINDOWS_ANES)
rows = []
for (rule, art, k, s), r in A.items():
    rows.append(dict(rule=rule, artefact=art, k_noise=k, sub=s, art_end_ms=round(r["art_end"], 2),
                     noise_uV=round(r["sd"], 3), n_peaks=len(r["lats"]),
                     latencies_ms=np.round(r["lats"], 2).tolist()))
peaks = pd.DataFrame(rows)
print(peaks.to_string())
print("\nrats with three peaks, by rule, artefact reading and margin:")
print(peaks.assign(three=peaks.n_peaks == 3).groupby(["rule", "artefact", "k_noise"]).three.sum().unstack().to_string())

In [ ]:
# Mean trace at the highest current per acute rat, with the peaks each rule assigns
# (artefact a, k = 3) and the paper's P1 to P3 ranges shaded.
fig, axs = plt.subplots(1, 5, figsize=(22, 4.5))
cols = {"first": "C0", "largest": "C1", "window": "C2", "inv": "C3"}
for ax, s in zip(axs, P.ACUTE):
    r = A[("first", "a", 3.0, s)]
    L, t = r["levels"], r["t"]
    y = (L[-1][1] + L[-1][2]) / 2
    y = y - P.baseline(y, t)
    ax.plot(t, y, "k", lw=0.8)
    for (lo, hi), c in zip(P.WINDOWS_ANES, ["#ddd", "#bbb", "#999"]):
        ax.axvspan(lo, hi, color=c, alpha=0.3)
    for j, (rule, c) in enumerate(cols.items()):
        for lat in A[(rule, "a", 3.0, s)]["lats"]:
            ax.axvline(lat, color=c, ls="--", lw=1, ymax=0.95 - 0.07 * j)
    sel = t > 1.0
    ax.set_xlim(0.6, 7.0); ax.set_ylim(y[sel].min() * 1.1, y[sel].max() * 1.2)
    ax.set_title(f"{s}, {L[-1][0]:.2f} mA\ny range {y[sel].min():.2f} to {y[sel].max():.2f} uV after 1 ms", fontsize=10)
    ax.set_xlabel("ms from pulse onset")
axs[0].set_ylabel("uV"); axs[0].legend([plt.Line2D([], [], color=c, ls="--") for c in cols.values()], list(cols), fontsize=8)
plt.tight_layout(); plt.show()
print("the artefact before 1 ms falls outside the y axis; the y range is that of the trace after 1 ms")

### anes-ecap-recorded

> In all five urethane-anaesthetised rats, ECAPs with complex waveforms were recorded on the
> non-stimulating electrode pair during graded stimulation and could be separated from the
> stimulation artefact.

Criterion: in each rat at least one positive peak after the artefact end reaches 0.05 uV in
both recordings at some current, and is absent at 0 mA. The lowest current in the files is
`CL0` (0.0175 mA) in four rats; 12-007 has none below 0.107 mA.

The files hold no 0 mA trace for the acute rats, so "absent at 0 mA" is read at the lowest
current in each file (an assumption: it is the nearest to 0 mA the data allow, and a peak absent
there would be absent at 0 mA too unless the trace at that current already holds a response).
That current is 17.5 uA in four rats and 106.5 uA in 12-007.

In [ ]:
rows = []
for rule in P.RULES:
    for art in "ab":
        for s in P.ACUTE:
            r = A[(rule, art, 3.0, s)]
            L, t = r["levels"], r["t"]
            ok_any = []
            for lat in r["lats"]:
                amps = [(min(P.amp_at(r1, t, lat, "a")[0], P.amp_at(r2, t, lat, "a")[0])) for _, r1, r2 in L]
                passes_some = any(a >= P.THRESH_UV for a in amps[1:])
                absent_low = amps[0] < P.THRESH_UV
                ok_any.append(dict(lat=round(lat, 2), low_uV=round(amps[0], 3), top_uV=round(amps[-1], 3),
                                   passes=passes_some, absent_lowest=absent_low))
            good = [o for o in ok_any if o["passes"] and o["absent_lowest"]]
            rows.append(dict(rule=rule, artefact=art, sub=s, lowest_mA=round(L[0][0], 4), peaks=ok_any,
                             n_peaks_meeting=len(good), meets=len(good) > 0))
rec = pd.DataFrame(rows)
for _, r in rec[rec.rule == "first"].iterrows():
    print(r.artefact, r["sub"], "lowest mA", r.lowest_mA, "meets:", r.meets, "|",
          "; ".join(f"{o['lat']} ms low {o['low_uV']} top {o['top_uV']} uV" for o in r.peaks))
print("\nrats meeting the criterion (k = 3), by rule and artefact reading:")
print(rec.groupby(["rule", "artefact"]).meets.sum().unstack().to_string())

**Result: reproduced.** Under the main rule, in all five acute rats at least one detected
positive peak after the artefact end reaches 0.05 uV in both recordings at some current and
stays below 0.05 uV at the lowest current, under both artefact-end readings (5 of 5 each). The
`largest` and `inv` rules also give 5 of 5; the `window` rule gives 2 of 5, as it finds a
triple inside the paper's ranges in two rats only. The lowest current, which stands for 0 mA
(an assumption), is 0.0175 mA in four rats and 0.107 mA in 12-007. Top-current amplitudes of
the peaks under the main rule are 0.07 to 15.7 uV. The outcome rests on the assumed peak
identification and on the lowest current standing for 0 mA.

### anes-ecap-latencies

> In anaesthetised rats, ECAPs at supra-threshold stimulation show three positive peaks centred
> at 1.80 ms (P1, range 1.65 to 2.90 ms), 3.48 ms (P2, 2.51 to 4.16 ms) and 3.77 ms (P3, 3.25 to
> 5.67 ms), n = 5.

Every combination of the criterion's readings: central value (median, mean), time origin
(pulse onset, end of the 250 us pulse), stimulus level (highest current, lowest current where
all three peaks pass, mean over currents at or above the P3 threshold), for both artefact-end
readings and three detection margins.

In [ ]:
EXP_LAT = np.array([1.80, 3.48, 3.77]); TOL_LAT = 0.1 * EXP_LAT
RANGE_LAT = P.WINDOWS_ANES
rows = []
per_rat = []
for rule in P.RULES:
    for art in "ab":
        for k in (2.0, 3.0, 5.0):
            for lev in "abc":
                for org in "ab":
                    M = np.array([A[(rule, art, k, s)]["lat"][(lev, org)] for s in P.ACUTE])
                    okr = ~np.isnan(M).any(1)
                    for cen, f in (("median", np.median), ("mean", np.mean)):
                        c = f(M[okr], 0) if okr.any() else np.full(3, np.nan)
                        rows.append(dict(rule=rule, artefact=art, k=k, level=lev, origin=org, central=cen,
                                         n_rats=int(okr.sum()), P1=c[0], P2=c[1], P3=c[2],
                                         within=bool(np.all(np.abs(c - EXP_LAT) <= TOL_LAT + 1e-9))))
                    if k == 3.0 and art == "a":
                        for s, m in zip(P.ACUTE, M):
                            per_rat.append(dict(rule=rule, level=lev, origin=org, sub=s, P1=m[0], P2=m[1], P3=m[2]))
lat_tab = pd.DataFrame(rows)
print(lat_tab.round(3).to_string())
print("combinations with all three within 10%:", int(lat_tab.within.sum()), "of", len(lat_tab))
print("by rule:", lat_tab.groupby("rule").within.agg(["sum", "size"]).to_dict("index"))
print(lat_tab[lat_tab.within].round(3).to_string())
main = lat_tab[(lat_tab.rule == "first") & (lat_tab.artefact == "a") & (lat_tab.k == 3.0) & (lat_tab.level == "a") & (lat_tab.origin == "a")]
print(main.round(3).to_string())
pr = pd.DataFrame(per_rat)
for rule in P.RULES:
    q = pr[(pr.rule == rule) & (pr.level == "a") & (pr.origin == "a")]
    print(f"\nrule {rule}, per rat (artefact a, k 3, highest current, onset origin):")
    print(q.round(2).to_string())
    for j, (lo, hi) in enumerate(RANGE_LAT):
        v = q[f"P{j+1}"].dropna()
        print(f"P{j+1}: per-rat range {v.min():.2f} to {v.max():.2f} ms; inside widened paper range "
              f"{lo*0.9:.2f} to {hi*1.1:.2f}: {int(((v >= lo*0.9) & (v <= hi*1.1)).sum())} of {len(v)}")
# distance of the closest combination, per rule
lat_tab["max_rel_err"] = (np.abs(lat_tab[["P1", "P2", "P3"]].values - EXP_LAT) / EXP_LAT).max(1)
for rule in P.RULES:
    print(f"closest combinations, rule {rule}:\n", lat_tab[lat_tab.rule == rule].sort_values("max_rel_err").head(3).round(3).to_string())

**Result: not reproduced.** None of the 288 combinations of readings (72 per peak rule, four
rules) puts all three central latencies within 10% of 1.80, 3.48 and 3.77 ms. Under the main
reading (rule `first`, artefact end at 1 ms, margin 3 SD, highest current, onset origin) the
medians over four rats are 1.60, 2.50 and 4.19 ms: P1 is 11% shorter, P2 28% shorter and P3
11% longer. The closest combination over all rules (rule `largest`, margin 5 SD, mean over
three rats) gives 1.79, 3.24 and 4.22 ms, 11.9% off at worst; under rule `first` the closest
gives 1.61, 2.98 and 4.18 ms (14.5%). Per rat under the main reading, P2 and P3 lie within the
paper's ranges widened by 10% (4 of 4 each) and P1 in 2 of 4. The outcome is the same under
every peak rule tried, and rests on the assumption that one of them is the paper's.

### anes-conduction-velocity

> Approximate conduction velocities of the three populations in anaesthetised rats are 1.60 m/s
> (P1, 0.98 to 1.72), 0.82 m/s (P2, 0.35 to 1.13) and 0.78 m/s (P3, 0.50 to 0.88), in the range
> of myelinated A-fibres and unmyelinated C-fibres.

Velocity per rat is distance over latency (mm/ms = m/s), with the latencies of
anes-ecap-latencies; the median or mean over rats, for each distance reading.

In [ ]:
EXP_V = np.array([1.60, 0.82, 0.78]); TOL_V = 0.1 * EXP_V
rows = []
for rule in P.RULES:
    for art in "ab":
        for k in (2.0, 3.0, 5.0):
            for lev in "abc":
                for org in "ab":
                    M = np.array([A[(rule, art, k, s)]["lat"][(lev, org)] for s in P.ACUTE])
                    okr = ~np.isnan(M).any(1)
                    if not okr.any():
                        continue
                    for dname, d in P.DISTANCES_MM.items():
                        V = d / M[okr]
                        for cen, f in (("median", np.median), ("mean", np.mean)):
                            c = f(V, 0)
                            rows.append(dict(rule=rule, artefact=art, k=k, level=lev, origin=org, distance=dname,
                                             central=cen, n=int(okr.sum()), P1=c[0], P2=c[1], P3=c[2],
                                             within=bool(np.all(np.abs(c - EXP_V) <= TOL_V + 1e-9))))
cv = pd.DataFrame(rows)
print("combinations with all three velocities within 10%:", int(cv.within.sum()), "of", len(cv))
print("by rule:", cv.groupby("rule").within.agg(["sum", "size"]).to_dict("index"))
print(cv[cv.within].round(3).to_string())
main = cv[(cv.rule == "first") & (cv.artefact == "a") & (cv.k == 3.0) & (cv.level == "a") & (cv.origin == "a")]
print(main.round(3).to_string())
cv["max_rel_err"] = (np.abs(cv[["P1", "P2", "P3"]].values - EXP_V) / EXP_V).max(1)
print("closest combinations:\n", cv.sort_values("max_rel_err").head(5).round(3).to_string())
M = np.array([A[("first", "a", 3.0, s)]["lat"][("a", "a")] for s in P.ACUTE])
print("per-rat velocities at 2.85 mm, rule first, highest current, onset origin:")
print(pd.DataFrame(2.85 / M, index=P.ACUTE, columns=["P1", "P2", "P3"]).round(2).to_string())

**Result: not reproduced.** None of the 864 combinations (216 per peak rule) puts all three
velocities within 10%. With 2.85 mm and the main latency reading, the medians are 1.82, 1.14
and 0.68 m/s against 1.60, 0.82 and 0.78 m/s; the closest combination (rule `largest`, margin
5 SD, mean over three rats) gives 1.57, 0.93 and 0.68 m/s, 13.1% off at worst. The velocities
do lie in the ranges the Discussion gives for A-fibres (1.6 to 21 m/s, P1) and C-fibres (0.5 to
1.6 m/s, P2 and P3) under the main reading.

### anes-threshold-order

> In anaesthetised rats the stimulation threshold rises in the order of the first (P1, 377 uA,
> 267 to 783), second (P2, 648 uA, 595 to 648) and third (P3, 930 uA, 550 to 1225) responding
> populations, a main effect in a Friedman test (P = 0.039, n = 5).

Threshold per rat and peak: the lowest current at which the amplitude is at least 0.05 uV in
both recordings. Readings: amplitude (a) above baseline, (b) above the preceding trough; step
(a) the lowest passing current, (b) interpolated. The files' currents are the logarithmic steps
listed in the data check, so reading (a) of the step is the lowest passing file current. A
further reading of "threshold" is shown: (s) the lowest current from which every higher current
also passes. Rats without three detected peaks drop out of the Friedman test.

In [ ]:
EXP_T = np.array([377, 648, 930.]); TOL_T = 0.1 * EXP_T
def thr_stats(M):
    okr = ~np.isnan(M).any(1)
    D = M[okr]
    if len(D) >= 3 and np.ptp(D) > 0:
        chi, p, rbar, pairs = P.dunn_friedman(D)
    else:
        chi, p, rbar, pairs = np.nan, np.nan, [np.nan] * 3, {(0, 2): (np.nan, np.nan)}
    return D, chi, p, pairs
rows = []
THR = {}
for rule in P.RULES:
    for art in "ab":
        for k in (2.0, 3.0, 5.0):
            for amp in "ab":
                for step in "ab":
                    for sus in (False, True):
                        M = np.array([A[(rule, art, k, s)]["thr"][(amp, step, sus)] if len(A[(rule, art, k, s)]["lats"]) == 3
                                      else [np.nan] * 3 for s in P.ACUTE], float)
                        THR[(rule, art, k, amp, step, sus)] = M
                        D, chi, p, pairs = thr_stats(M)
                        for cen, f in (("median", np.median), ("mean", np.mean)):
                            c = f(D, 0) if len(D) else np.full(3, np.nan)
                            rows.append(dict(rule=rule, artefact=art, k=k, amp=amp, step=step, sustained=sus,
                                             central=cen, n=len(D), P1=c[0], P2=c[1], P3=c[2],
                                             rising=bool(c[0] < c[1] < c[2]), friedman_chi2=chi, friedman_p=p,
                                             dunn_p31=pairs[(0, 2)][1], dunn_z31=pairs[(0, 2)][0],
                                             within=bool(np.all(np.abs(c - EXP_T) <= TOL_T + 1e-9))))
thr_tab = pd.DataFrame(rows)
thr_tab["meets"] = thr_tab.rising & (thr_tab.friedman_p < 0.05)
print(thr_tab.round(4).to_string())
print("combinations meeting the criterion (rising and Friedman P < 0.05):", int(thr_tab.meets.sum()), "of", len(thr_tab))
print("by rule:", thr_tab.groupby("rule").meets.agg(["sum", "size"]).to_dict("index"))
print(thr_tab[thr_tab.meets].round(4).to_string())
print("combinations with all three values within 10%:", int(thr_tab.within.sum()))
print("smallest Friedman P with rising central values, by rule:",
      thr_tab[thr_tab.rising].groupby("rule").friedman_p.min().round(4).to_dict())
for rule in P.RULES:
    print(f"\nper-rat thresholds (uA), rule {rule}, artefact a, k = 3, amplitude a, step a, first passing:")
    print(pd.DataFrame(THR[(rule, "a", 3.0, "a", "a", False)], index=P.ACUTE, columns=["P1", "P2", "P3"]).round(0).to_string())
print("per-rat thresholds (uA), rule first, sustained reading:")
print(pd.DataFrame(THR[("first", "a", 3.0, "a", "a", True)], index=P.ACUTE, columns=["P1", "P2", "P3"]).round(0).to_string())
print("lowest file current, uA:", [round(A[("first", "a", 3.0, s)]["min_mA"] * 1000, 1) for s in P.ACUTE])

Variants outside the criterion: the paper's 0.05 uV is within the noise of most files, so the
same thresholds are computed with the pass level raised (i) to three times each file's noise SD
(where that is above 0.05 uV), and (ii) to 5 uV in the stored units, which is 0.05 uV if the
stored values carry the 10^2 gain of the Methods. Both are put through the Friedman and Dunn
tests, for each rule (artefact a, k = 3, amplitude a, lowest passing current).

In [ ]:
rows, vrows = [], []
VAR = {}
for rule in P.RULES:
    for vname in ("3 noise SD", "5 uV (gain 100)"):
        Ms = []
        for s in P.ACUTE:
            r = A[(rule, "a", 3.0, s)]
            lvl = max(0.05, 3 * r["sd"]) if vname == "3 noise SD" else 5.0
            old = P.THRESH_UV
            P.THRESH_UV = lvl
            try:
                th = P.thresholds(r["levels"], r["t"], r["lats"], "a", "a") if len(r["lats"]) else []
            finally:
                P.THRESH_UV = old
            rows.append(dict(rule=rule, variant=vname, sub=s, pass_uV=round(lvl, 3), thr_uA=np.round(th, 0).tolist()))
            Ms.append(th if len(th) == 3 else [np.nan] * 3)
        M = np.array(Ms, float)
        VAR[(rule, vname)] = M
        D, chi, p, pairs = thr_stats(M)
        med = np.median(D, 0) if len(D) else np.full(3, np.nan)
        vrows.append(dict(rule=rule, variant=vname, n=len(D), P1=med[0], P2=med[1], P3=med[2],
                          rising=bool(med[0] < med[1] < med[2]), friedman_chi2=chi, friedman_p=p,
                          dunn_z31=pairs[(0, 2)][0], dunn_p31=pairs[(0, 2)][1],
                          within=bool(np.all(np.abs(med - EXP_T) <= TOL_T + 1e-9))))
print(pd.DataFrame(rows).to_string())
var_tab = pd.DataFrame(vrows)
print(var_tab.round(4).to_string())

How often does noise alone pass the paper's test? Band-passed Gaussian noise (500 to 3000 Hz,
4th-order Butterworth, as the files' filter; 200 kHz; 8 ms), two independent recordings, at the
SDs of the deposited files; the peak nearest a latency is measured as in `amp_at`. The
generator is seeded from the claim (`payne-2020/anes-threshold-order`), with the four further
seeds of the protocol.

In [ ]:
import zlib
from scipy import signal as ssig
b_, a_ = ssig.butter(4, [500, 3000], btype="band", fs=200000)
tt_ = 0.6 + np.arange(1600) / 200.0
def noise_trace(rng, sd):
    x = ssig.lfilter(b_, a_, rng.standard_normal(4000))[-1600:]
    return x / x.std() * sd
seed0 = zlib.crc32(b"payne-2020/anes-threshold-order")
rows = []
for off in (0, 1000, 2000, 3000, 4000):
    rng = np.random.default_rng(seed0 + off)
    for sd in (0.02, 0.05, 0.2, 0.7):
        n_pass = 0
        for _ in range(400):
            r1, r2 = noise_trace(rng, sd), noise_trace(rng, sd)
            n_pass += min(P.amp_at(r1, tt_, 3.0, "a")[0], P.amp_at(r2, tt_, 3.0, "a")[0]) >= P.THRESH_UV
        rows.append(dict(seed_offset=off, noise_sd_uV=sd, false_pass=n_pass / 400))
fp = pd.DataFrame(rows).pivot(index="noise_sd_uV", columns="seed_offset", values="false_pass")
print("fraction of noise-only current steps passing the 0.05 uV test in both recordings, by seed:")
print(fp.round(3).to_string())

**Result: not reproduced.** Under the criterion's threshold (the lowest current that passes),
no combination of readings under any peak rule (0 of 192) gives thresholds rising from P1 to
P3 with a Friedman P below 0.05, and none puts the three values within 10% of 377, 648 and 930
uA. Under the main reading the medians over four rats are 62, 221 and 280 uA, rising, with
Friedman chi2 = 4.5 and P = 0.105. Twelve combinations of 384 meet the criterion, all under
the further "sustained" reading (lowest current from which every higher current passes), which
is not the criterion's definition: rule `largest` with artefact end (b) and margin 3 (Friedman
P = 0.022 and 0.039), and rule `window` with margin 2 (P = 0.039); none is within tolerance.
The 0.05 uV level lies within the noise of most files: noise alone passes the two-recording
test in 24 to 28% of current steps at 0.05 uV SD and 63 to 77% at 0.2 to 0.7 uV SD (five
seeds), so several thresholds equal the lowest current. With the pass level raised to three
noise SDs (a variant outside the criterion) the main rule gives medians of 107, 498 and 691
uA, rising, Friedman chi2 = 6.5, P = 0.039 (n = 4), still outside tolerance; with a pass level
of 5 uV (the stored values read as carrying a gain of 100) at most one rat has three
thresholds.

### fig2c-p3-above-p1

> The third responding population has a significantly higher threshold than the first (P = 0.034).

Dunn's test on the Friedman mean ranks, P3 against P1, unadjusted and Bonferroni-adjusted over
three pairs, for every threshold reading of anes-threshold-order.

In [ ]:
d = thr_tab[thr_tab.central == "median"].copy()
d["dunn_p31_bonf"] = np.minimum(1, 3 * d.dunn_p31)
d["P3_higher"] = d.dunn_z31 > 0
cols_ = ["rule", "artefact", "k", "amp", "step", "sustained", "n", "friedman_p", "dunn_z31", "dunn_p31", "dunn_p31_bonf", "P3_higher"]
print(d[cols_].round(4).to_string())
for rule in P.RULES:
    q = d[d.rule == rule]
    print(f"rule {rule}: readings with P3 higher and unadjusted P < 0.05:", int(((q.dunn_p31 < 0.05) & q.P3_higher).sum()), "of", len(q),
          "; Bonferroni P < 0.05:", int(((q.dunn_p31_bonf < 0.05) & q.P3_higher).sum()))
ok = d[(d.dunn_p31 < 0.05) & d.P3_higher]
print("passing readings:\n", ok[cols_].round(4).to_string())
print("smallest unadjusted Dunn P with P3 higher: %.4f" % d[d.P3_higher].dunn_p31.min())
print("\nwith all five rats (rows with n = 5), smallest unadjusted P with P3 higher: %.4f" % d[d.P3_higher & (d.n == 5)].dunn_p31.min())
v = var_tab.copy(); v["dunn_p31_bonf"] = np.minimum(1, 3 * v.dunn_p31)
print("variants with a raised pass level:\n", v.round(4).to_string())

**Result: reproduced, under one reading.** Under the main threshold reading Dunn's test on the
Friedman ranks gives P3 above P1 with z = 2.12 and unadjusted P = 0.034 (n = 4 rats; 12-008 has
no third peak); Bonferroni-adjusted over three pairs it is P = 0.10. Of the 48 readings per
peak rule, P3 is higher with unadjusted P below 0.05 in 4 (rule `first`), 4 (`largest`), 8
(`window`) and 9 (`inv`); with the Bonferroni P in 0, 2, 4 and 0, all of these under the
sustained reading. With all five rats in the test (margin 2 SD) the smallest unadjusted P is
0.082. The Friedman test in the main reading is not significant (P = 0.105), so the post hoc
result comes without the main effect the paper reports; under the three-noise-SD variant both
are significant (Friedman P = 0.039, Dunn P = 0.013). The pass is narrow: it rests on the
assumed peak identification, on n = 4, and on the unadjusted reading of Dunn's P.

### fig2ab-one-rat-p2-below-p1

> In four of five anaesthetised rats the first responding population had a lower threshold than
> later responding populations; in one rat the threshold of P1 was higher than that of P2.

Per rat: P1 below both P2 and P3, and P1 above P2, under each threshold reading. Ties (equal
thresholds) count as neither.

In [ ]:
rows = []
for key, M in THR.items():
    rule, art, k, amp, step, sus = key
    lower = int(np.sum((M[:, 0] < M[:, 1]) & (M[:, 0] < M[:, 2])))
    higher = int(np.sum(M[:, 0] > M[:, 1]))
    rows.append(dict(rule=rule, artefact=art, k=k, amp=amp, step=step, sustained=sus,
                     rats_with_3_peaks=int((~np.isnan(M).any(1)).sum()),
                     p1_lowest=lower, p1_above_p2=higher, ties_p1_p2=int(np.sum(M[:, 0] == M[:, 1])),
                     meets=(lower == 4 and higher == 1)))
for (rule, vname), M in VAR.items():
    lower = int(np.sum((M[:, 0] < M[:, 1]) & (M[:, 0] < M[:, 2])))
    higher = int(np.sum(M[:, 0] > M[:, 1]))
    rows.append(dict(rule=rule, artefact="a", k=3.0, amp="a", step="a", sustained=False, variant=vname,
                     rats_with_3_peaks=int((~np.isnan(M).any(1)).sum()),
                     p1_lowest=lower, p1_above_p2=higher, ties_p1_p2=int(np.sum(M[:, 0] == M[:, 1])),
                     meets=(lower == 4 and higher == 1)))
o = pd.DataFrame(rows)
print(o.to_string())
print("readings meeting the criterion:", int(o[o.variant.isna()].meets.sum()), "of", int(o.variant.isna().sum()),
      "; variants with a raised pass level:", int(o[o.variant.notna()].meets.sum()), "of", int(o.variant.notna().sum()))

**Result: not reproduced.** Under the criterion's threshold (lowest passing current) no reading
under any peak rule gives four of five rats with P1 lowest and one with P1 above P2 (0 of 96
non-sustained rows). Under the main reading three of four rats have P1 lowest and one (12-005)
has P1 above P2 (648 against 541 uA), the pattern of the legend in four rats; 12-008 has no
third peak. Two rows of 192 meet the counts, both under rule `largest`, artefact end (b),
margin 2 SD and the sustained reading, which is not the criterion's definition of threshold;
the raised pass levels give 0 of 8.

## Awake ECAPs

### awake-ecap-latencies

> In chronically implanted awake rats (n = 4), three populations were recruited in all animals
> at week 0 and week 1, with mean latencies of 2.27 ms (P1, 1.93 to 2.65), 3.92 ms (P2, 3.19 to
> 4.44) and 5.62 ms (P3, 4.57 to 6.72) and conduction velocities of 1.26, 0.73 and 0.51 m/s.

The seven `chronic ECAP` files, four at day 0 (12-050, 12-051, 12-052, 12-056) and three at day
7 (12-053, 12-055, 12-057). Same peak detection as the anaesthetised rats. Reading (b) of the
criterion (mean over rats of each rat's two-week mean) needs the week 0 and week 1 recordings
of the same rat, which the files do not link; readings (a), all seven recordings, and (c), week
0 alone, are run.

In [ ]:
AW = {}
for rule in P.RULES:
    for art in "ab":
        for k in (2.0, 3.0, 5.0):
            for s in P.CHRONIC:
                AW[(rule, art, k, s)] = P.analyse_ecap(s, art=art, k_noise=k, rule=rule, windows=P.WINDOWS_AWAKE)
DAY = sj.set_index("subject_id")["Recording Day (post-surgery)"]
rows = []
for (rule, art, k, s), r in AW.items():
    rows.append(dict(rule=rule, artefact=art, k=k, sub=s, day=int(DAY["sub-" + s]), noise_uV=round(r["sd"], 3),
                     n_peaks=len(r["lats"]), latencies=np.round(r["lats"], 2).tolist()))
aw = pd.DataFrame(rows)
print(aw.to_string())
print("\nrecordings with three peaks, by reading:")
print(aw.assign(three=aw.n_peaks >= 3).groupby(["rule", "artefact", "k"]).three.sum().unstack().to_string())
EXP_AL = np.array([2.27, 3.92, 5.62]); EXP_AV = np.array([1.26, 0.73, 0.51])
rows = []
for rule, art, k in [(r_, a_, k_) for r_ in P.RULES for a_ in "ab" for k_ in (2.0, 3.0, 5.0)]:
        for lev in "abc":
            for org in "ab":
                for rd, subs in (("a: all 7", P.CHRONIC), ("c: week 0", [s for s in P.CHRONIC if DAY["sub-" + s] == 0])):
                    M = np.array([AW[(rule, art, k, s)]["lat"][(lev, org)] for s in subs])
                    okr = ~np.isnan(M).any(1)
                    if not okr.any():
                        continue
                    c = M[okr].mean(0)
                    for dname, dmm in P.DISTANCES_MM.items():
                        v = (dmm / M[okr]).mean(0)
                        rows.append(dict(rule=rule, artefact=art, k=k, level=lev, origin=org, reading=rd,
                                         n=int(okr.sum()), of=len(subs), distance=dname,
                                         P1=c[0], P2=c[1], P3=c[2], v1=v[0], v2=v[1], v3=v[2],
                                         lat_within=bool(np.all(np.abs(c - EXP_AL) <= 0.1 * EXP_AL + 1e-9)),
                                         v_within=bool(np.all(np.abs(v - EXP_AV) <= 0.1 * EXP_AV + 1e-9))))
awl = pd.DataFrame(rows)
print(awl.round(3).to_string())
print("rows with all six values within tolerance:", int((awl.lat_within & awl.v_within).sum()), "of", len(awl))
print("rows with the three latencies within tolerance:", int(awl.lat_within.sum()), "; with the three velocities:", int(awl.v_within.sum()))
print("by rule (all six within):", awl.assign(both=awl.lat_within & awl.v_within).groupby("rule").both.agg(["sum", "size"]).to_dict("index"))
print("most recordings with three peaks under any reading:", int(aw.assign(three=aw.n_peaks >= 3).groupby(["rule", "artefact", "k"]).three.sum().max()), "of 7")
# velocity as distance over the mean latency (how the paper's printed velocities relate to its latencies)
dist = awl.distance.map(P.DISTANCES_MM).values[:, None]
vml = dist / awl[["P1", "P2", "P3"]].values
awl["v_of_mean_within"] = np.all(np.abs(vml - EXP_AV) <= 0.1 * EXP_AV + 1e-9, axis=1)
print("velocity as distance over the mean latency: rows with the three velocities within tolerance:",
      int(awl.v_of_mean_within.sum()), "; with all six values:", int((awl.lat_within & awl.v_of_mean_within).sum()), "of", len(awl))
sel = awl[awl.lat_within & awl.v_of_mean_within]
print(sel.assign(v1m=vml[sel.index, 0], v2m=vml[sel.index, 1], v3m=vml[sel.index, 2])
      [["rule", "artefact", "k", "level", "origin", "reading", "n", "of", "distance", "P1", "P2", "P3", "v1m", "v2m", "v3m"]].round(3).to_string())

**Result: not reproduced.** Three peaks are found in at most 5 of the 7 awake recordings under
any reading (margin 2 SD) and in 2 of 7 under the main reading, where the claim has three peaks
in every rat at both weeks; 12-057 shows none under any rule or margin. No
combination of readings puts all six values within tolerance (0 of 792). Under rule `largest`
with margin 2 SD the mean latencies over the five recordings with three peaks (2.14, 3.70 and
5.28 ms at the highest current) are within tolerance, while the mean velocities (1.59, 0.88 and
0.57 m/s at 2.85 mm, the mean of each recording's velocity) are not; 9 rows have the latencies
within tolerance and 23 the velocities, none both. That "none both" depends on how velocity is
averaged: taken as 2.85 mm over the mean latency, as the paper's printed velocities are, the same
latencies give 1.33, 0.77 and 0.54 m/s, within tolerance, and 3 of 792 rows (rule `largest`,
margin 2 SD, reading (a), each stimulus level, onset origin, 5 of 7 recordings) have all six
values within tolerance. The outcome stays, as these rows rest on 5 of 7 recordings and the
criterion needs three peaks in every rat. With rule `first`, margin 2 SD, onset origin and the highest current the
means are 1.94, 3.30 and 4.65 ms. The files hold four rats at day 0 and three at day 7 with no
link between them, so the readings over weeks per rat cannot be run.

### awake-threshold-increase (blocked)

> In awake implanted rats, thresholds of all three populations increased from week 0 to week 1
> (P1 391 to 643 uA, P2 677 to 1051 uA, P3 713 to 1263 uA), significantly by a Friedman test
> (P = 0.039, n = 4).

Blocked: every reading of the criterion's Friedman test has rat as a block, and no rat's week 0
and week 1 recordings can be paired (seven distinct subject identifiers). It also depends on
awake-ecap-latencies. The per-file thresholds are printed for reference only, where a file has
three detected peaks.

In [ ]:
rows = []
for s in P.CHRONIC:
    r = AW[("first", "a", 3.0, s)]
    rows.append(dict(sub=s, day=int(DAY["sub-" + s]), n_peaks=len(r["lats"]),
                     thr_uA=np.round(r["thr"][("a", "a", False)], 0).tolist(),
                     thr_sustained_uA=np.round(r["thr"][("a", "a", True)], 0).tolist(),
                     lowest_mA=r["min_mA"], highest_mA=r["max_mA"]))
print(pd.DataFrame(rows).to_string())

## Impedances

### impedance-increase

> Mean electrode impedance rose from 4.19 kOhm before implantation to 7.25 kOhm on day 1,
> 12.04 kOhm at 2 weeks and 13.1 kOhm at 8 weeks.

Two sources: the derivative `chronic-impedances.pzfx` (one value per array, with columns
"Saline impedance", "Post Surgery" and "Chronic implanation"), and the per-electrode CSVs of the
eight 18-* rats. For the CSVs, the "pre implant" row is before implantation, the first
"post surgery" or "surgery" row is day 1, and the last row is the chronic value (9 to 18 days
after surgery). No impedance file belongs to the two 8-week rats (15-759, 15-760).

In [ ]:
EXP_Z = [(4.19, 0.17, 3.16, 5.03), (7.25, 0.79, 4.54, 15.18), (12.04, 1.15, 6.24, 18.3), (13.1, 2.06, 8.0, 20.2)]
rows = []
for (col, v), (e, tol, lo, hi) in zip(pz.items(), EXP_Z):
    v = np.array(v) / 1000
    v = v[np.isfinite(v)]
    rows.append(dict(source="pzfx per array", stage=col, n=len(v), mean=v.mean(), sem=v.std(ddof=1) / np.sqrt(len(v)),
                     sd=v.std(ddof=1), min=v.min(), max=v.max(), paper=e, tol=tol,
                     within=abs(v.mean() - e) <= tol, range_ok=(v.min() >= lo * 0.9) and (v.max() <= hi * 1.1)))
imp["date"] = pd.to_datetime(imp.DATE, dayfirst=True)
stage_rows = {"pre": [], "day1": [], "chronic": []}
for a, g in imp.groupby("ANIMAL"):
    pre = g[g.CONTEXT.str.contains("pre implant")]
    post = g[g.CONTEXT.str.match(r"^(post )?surgery$")]
    stage_rows["pre"].append(pre[Z].values.ravel())
    stage_rows["day1"].append(post[Z].values.ravel()[:4])
    stage_rows["chronic"].append(g.sort_values("date").iloc[-1][Z].values.astype(float))
for (st, arrs), (e, tol, lo, hi) in zip(stage_rows.items(), EXP_Z):
    v = np.concatenate(arrs).astype(float) / 1000
    rows.append(dict(source="csv per electrode", stage=st, n=len(v), mean=v.mean(), sem=v.std(ddof=1) / np.sqrt(len(v)),
                     sd=v.std(ddof=1), min=v.min(), max=v.max(), paper=e, tol=tol,
                     within=abs(v.mean() - e) <= tol, range_ok=(v.min() >= lo * 0.9) and (v.max() <= hi * 1.1)))
zt = pd.DataFrame(rows)
print(zt.round(3).to_string())
print("8-week values: no file (the 8-week rats 15-759 and 15-760 have no impedance file)")
print("arrays in the CSVs:", imp.ANIMAL.nunique(), "; dates of the last row per array:",
      imp.sort_values("date").groupby("ANIMAL").date.last().dt.date.to_dict())

**Result: partially reproduced.** The derivative's per-array values give 4.19 kOhm (SEM 0.17,
12 arrays) before implantation, 7.25 (SEM 0.79, 12) after surgery and 12.04 (SEM 1.15, 10) at
about 2 weeks, equal to the paper's values and spreads, and their ranges (3.16 to 5.03, 4.54 to
15.18, 6.24 to 18.32 kOhm) equal the printed ranges. The 8-week value (13.1 kOhm, 2 arrays)
cannot be tested: no impedance file belongs to the 8-week rats. The paper's spreads are SEMs
over arrays, and its n (38 electrodes in 10 arrays) is not the 12 arrays averaged. The
per-electrode CSVs of eight rats give 3.89 (32 electrodes), 7.02 (28) and 20.2 kOhm (32,
including one open electrode at 252 kOhm). The paper's n of 38 electrodes in 10 arrays does not match the
pzfx, which holds 12 per-array values before implantation and on day 1 and 10 at about 2 weeks;
the means match all the same.

### electrode-failures (blocked)

> No short circuits occurred during implantation, and only 2 of 40 electrodes became open
> circuit, both within 2 weeks of implantation.

Blocked as `data_missing`: per-electrode impedances are in the dataset for 8 arrays (32
electrodes), not the paper's 40 electrodes; the per-array derivative cannot show single
electrodes. What the 32 show is printed for reference: values far above the array's others
(more than three times the array's median) and whether they recur.

In [ ]:
rows = []
for _, r in imp.iterrows():
    v = r[Z].values.astype(float)
    med = np.median(v)
    for e, x in zip(Z, v):
        if x > 3 * med:
            rows.append(dict(animal=r.ANIMAL, date=r.DATE, context=r.CONTEXT, electrode=e, ohm=x, array_median=med))
hi_ = pd.DataFrame(rows)
print(hi_.to_string())
print("electrodes with a value > 3x array median:", hi_.groupby(["animal", "electrode"]).size().to_dict())
print("lowest value in any CSV (Ohm):", imp[Z].min().min(), "; electrodes in the CSVs:", 4 * imp.ANIMAL.nunique())

## colonic-pressure

> Pelvic nerve stimulation increased colonic pressure in only 1 of 3 anaesthetised rats, by
> 2.6 mmHg (mean and SEM 2.6 and 0.1, 13 replicate responses), time-locked to stimulation.

Per train, the increase is (a) the peak colonic pressure during the train minus the mean over
an equal window before it, or (b) the mean during the train minus the mean before it. Each entry
of the stimulus table is taken as a train of the file's `StimDuration`. A rat responds when its
increases exceed zero by a one-sample t-test at P < 0.05 (one-sided, as the criterion asks for
an increase above zero; the two-sided P is printed too).

A variant outside the criterion: reading (a) takes a peak against a mean, so it is positive
even with no response. Each rat's increases are also compared with the same measure taken
halfway between trains (Welch one-sided t-test), which removes that bias.

In [ ]:
rows = []
per = []
for s in P.COLON:
    tt, p, St, sa = COL[s]
    D = float(sa["StimDuration"])
    for rd in "ab":
        inc = []
        for t0 in St:
            dur = (tt >= t0) & (tt < t0 + D)
            pre = (tt >= t0 - D) & (tt < t0)
            if pre.sum() == 0 or dur.sum() == 0:
                continue
            inc.append((np.nanmax(p[dur]) if rd == "a" else np.nanmean(p[dur])) - np.nanmean(p[pre]))
        inc = np.array(inc)
        t1 = stats.ttest_1samp(inc, 0, alternative="greater")
        t2 = stats.ttest_1samp(inc, 0)
        rows.append(dict(sub=s, reading=rd, trains=len(inc), mean=inc.mean(), sem=inc.std(ddof=1) / np.sqrt(len(inc)),
                         min=inc.min(), max=inc.max(), p_one_sided=t1.pvalue, p_two_sided=t2.pvalue,
                         responds=t1.pvalue < 0.05))
        # null: same measure at times halfway between trains
        mids = (St[:-1] + St[1:]) / 2
        nul = []
        for t0 in mids:
            dur = (tt >= t0) & (tt < t0 + D); pre = (tt >= t0 - D) & (tt < t0)
            if pre.sum() and dur.sum():
                nul.append((np.nanmax(p[dur]) if rd == "a" else np.nanmean(p[dur])) - np.nanmean(p[pre]))
        tn = stats.ttest_ind(inc, nul, equal_var=False, alternative="greater")
        per.append(dict(sub=s, reading=rd, null_mean=np.mean(nul), null_n=len(nul), excess=inc.mean() - np.mean(nul),
                        p_vs_null=tn.pvalue, responds_vs_null=tn.pvalue < 0.05))
col = pd.DataFrame(rows).merge(pd.DataFrame(per), on=["sub", "reading"])
print(col.round(4).to_string())
for rd in "ab":
    c = col[col.reading == rd]
    print(f"reading {rd}: responders {int(c.responds.sum())} of 3; responder means (mmHg)",
          c[c.responds][["sub", "mean", "sem", "trains"]].round(3).to_dict("records"))
    print(f"reading {rd}, against the between-train null (variant, Welch one-sided): responders {int(c.responds_vs_null.sum())} of 3;",
          c[c.responds_vs_null][["sub", "mean", "excess", "p_vs_null"]].round(4).to_dict("records"))

Variant outside the criterion: the train is taken as the `StimDuration` before each recorded
time (the clock check above shows the 15-758 response there).

In [ ]:
rows = []
for s in P.COLON:
    tt, p, St, sa = COL[s]
    D = float(sa["StimDuration"])
    for rd in "ab":
        inc = []
        for t1 in St:
            t0 = t1 - D
            dur = (tt >= t0) & (tt < t1); pre = (tt >= t0 - D) & (tt < t0)
            if pre.sum() and dur.sum():
                inc.append((np.nanmax(p[dur]) if rd == "a" else np.nanmean(p[dur])) - np.nanmean(p[pre]))
        inc = np.array(inc)
        t1_ = stats.ttest_1samp(inc, 0, alternative="greater")
        rows.append(dict(sub=s, reading=rd, trains=len(inc), mean=inc.mean(), sem=inc.std(ddof=1) / np.sqrt(len(inc)),
                         p_one_sided=t1_.pvalue, responds=t1_.pvalue < 0.05))
print(pd.DataFrame(rows).round(4).to_string())

**Result: not reproduced.** Under reading (a) (peak during the train minus the mean before)
all three rats respond (mean increases 5.68, 4.26 and 0.25 mmHg; P = 0.019, 0.006 and 0.0002),
but the same measure halfway between trains is 5.48, 2.51 and 0.56 mmHg, so reading (a) is
biased upwards; against that null (variant) no rat responds under either reading. Under
reading (b) (mean during minus mean before) one rat responds, 12-020, with 1.44 mmHg (SEM 0.70,
14 trains), outside 2.6 +- 0.1. With the train placed in the 10 s before each recorded time
(variant), reading (b) again gives one responder, 15-758, with 0.64 mmHg (SEM 0.09, 33 trains).
No reading gives a responder near 2.6 mmHg, and no rat has 13 trains.

### no-fecal-expulsion-or-erection (blocked)

Blocked as `data_missing`: the dataset holds no observation notes, video or annotation of
faecal pellets or erection (data check: no such file in the manifest).

## Table 1: unstimulated voiding

The derivative `cystometry-dataset.xlsx` holds the three pre-stimulation cycles of nine rats
(the paper's n = 9) for every Table 1 parameter; the dataset's own R notebook
(`files/code/cystometry-analysis.rmd`) summarises it as the mean of each rat's mean with a
t-based 95% CI. The criterion's readings: (a) mean of rat means with a t-based 95% CI, (b) the
mean over all cycles pooled. Rat 18-273's third cycle has no contraction duration; reading (a)
is shown with that rat's mean over its two other cycles (a1) and with the rat left out (a2),
which is what the R code's `mean()` does with a missing value. Threshold pressure and
contraction duration are taken as the derivative defines them (`mmhg_init`,
`contraction_dur_s`); the readings of their definitions in the criterion cannot be told apart
from the derivative's values, and the primary `/Voids` tables do not match the derivative (data
check), so the derivative is used as the authors computed it. The next cell prints the R code's summary
chunk and the summary table in its output (`files/derivative/cystometry-analysis.nb.html`).

In [ ]:
import html, json as _json
rmd = P.read_bytes("files/code/cystometry-analysis.rmd").decode()
i = rmd.find("### Table: Summary statistics"); print(rmd[i:rmd.find("```", rmd.find("fn_summary(df_voids)")) + 3])
nbh = P.read_bytes("files/derivative/cystometry-analysis.nb.html").decode(errors="replace")
j = nbh.find("{\"columns\"", nbh.find("fn_summary(df_voids)"))
obj, _ = _json.JSONDecoder().raw_decode(html.unescape(nbh[j:j + 50000]))
rsum = pd.DataFrame(obj["data"]).rename(columns={str(c["name"][0]): c["label"][0] for c in obj["columns"]})
print("summary table in files/derivative/cystometry-analysis.nb.html:")
print(rsum.to_string())

Table 1 recomputed from the primary pressure traces. The derivative table gives one
definition of each parameter, and which reading of the criterion it matches is not stated.
Here each parameter is recomputed from the `/Cystometry` traces of the same nine rats under the
criterion's readings (`pelvic.void_params`): threshold at the onset of the rise (a) or at the
start of the HFPOs, the proxy for urine release (b); contraction duration from onset to the
return to the minimum (a) or the time above threshold (b); HFPO frequency from the spectrum
peak (a) or the count of oscillation peaks per second (b). The onset of each contraction is the
authors' `/Voids` ThresholdTime marker; the inter-void interval is the time between successive
onsets within one recorded wave. The minimum is the lowest 1 s mean pressure from the peak to
the next onset, or (second reading) within 60 s of the peak; relative pressures are given
against both. Two selections of cycles: the first three `/Voids` entries before the first
stimulation train ("pre-stimulation", the Methods' "three cycles prior to the delivery of
stimulation"), and the first three entries ("first three"), as some rats have fewer than three
marked voids before their first train.

In [ ]:
rows = []
for s in sorted({to_sub(x) for x in dv.subject_id}):
    W, V, S = CY[s]
    V = V.sort_values("ThresholdTime").reset_index(drop=True)
    first_train = np.inf if S is None else float(np.min(S["Time"].values if "Time" in S else [S.attrs["Time"]]))
    for sel, ks in (("pre-stimulation", [k for k in range(len(V)) if V.ThresholdTime[k] < first_train][:3]),
                    ("first three", list(range(min(3, len(V)))))):
        for k in ks:
            nxt = V.ThresholdTime[k + 1] if k + 1 < len(V) else np.inf
            d = P.void_params(W, V, k, next_onset=nxt)
            d.update(sub=s, selection=sel, void=k,
                     ivi_min=P.ivi_min(W, V, k))
            rows.append(d)
TR = pd.DataFrame(rows)
print(TR.round(2).to_string())
print("cycles per rat:", TR.groupby(["selection", "sub"]).size().unstack(0).to_dict())
print("cycles without an HFPO period (thr_b):", int(TR.thr_b.isna().sum()), "of", len(TR))

In [ ]:
T1 = {"table1-minimum-pressure": ("mmhg_eov", 10.3, 7.4, 13.2),
      "table1-threshold-pressure": ("mmhg_init_rel", 6.6, 4.7, 8.4),
      "table1-peak-pressure": ("mmhg_peak_rel", 51.2, 39.5, 63.0),
      "table1-contraction-duration": ("contraction_dur_s", 28.9, 17.1, 37.5),
      "table1-intervoid-interval": ("ivi_minute", 7.6, 5.9, 9.4),
      "table1-hfpo-frequency": ("eus_freq", 10.7, 8.3, 13.0)}
# the same parameters recomputed from the pressure traces (next section), per definitional reading
T1_TRACE = {"table1-minimum-pressure": ["minimum", "minimum_60"],
            "table1-threshold-pressure": ["thr_a_rel", "thr_b_rel", "thr_a_rel_60", "thr_b_rel_60"],
            "table1-peak-pressure": ["peak_rel", "peak_rel_60"],
            "table1-contraction-duration": ["dur_a", "dur_b"],
            "table1-intervoid-interval": ["ivi_min"],
            "table1-hfpo-frequency": ["hfpo_a", "hfpo_b"]}
ratm = dv.groupby(["parameter_id", "subject_id"]).value.mean().unstack(0)
ratm_drop = dv.groupby(["parameter_id", "subject_id"]).value.apply(lambda x: x.mean() if x.notna().all() else np.nan).unstack(0)
def t1(claim):
    pid, e, lo, hi = T1[claim]
    rows = []
    for rd, x in (("a1: rat means", ratm[pid]), ("a2: rat means, incomplete rats out", ratm_drop[pid])):
        m, l, h, n = P.mean_ci(x)
        rows.append(dict(reading=rd, n=n, mean=m, ci_low=l, ci_high=h, paper=e, paper_ci=(lo, hi),
                         inside_paper_ci=lo <= m <= hi))
    v = dv[dv.parameter_id == pid].value.dropna()
    m, l, h, n = P.mean_ci(v)
    rows.append(dict(reading="b: cycles pooled", n=n, mean=m, ci_low=l, ci_high=h, paper=e, paper_ci=(lo, hi),
                     inside_paper_ci=lo <= m <= hi))
    print(pd.DataFrame(rows).round(3).to_string())
    print("per-rat means:", ratm[pid].round(2).to_dict())
    print("rats:", ratm[pid].notna().sum(), "; cycles:", int(v.size), "; range of cycle values %.2f to %.2f" % (v.min(), v.max()))
    rows = []
    for sel in ("pre-stimulation", "first three"):
        q = TR[TR.selection == sel]
        for par in T1_TRACE[claim]:
            rm = q.groupby("sub")[par].mean()
            for rd, x in (("a1: rat means", rm), ("b: cycles pooled", q[par])):
                m, l, h, n = P.mean_ci(x)
                rows.append(dict(source="trace", selection=sel, definition=par, reading=rd, n=n, mean=m, ci_low=l,
                                 ci_high=h, inside_paper_ci=lo <= m <= hi))
    print("recomputed from the pressure traces:")
    print(pd.DataFrame(rows).round(3).to_string())

### table1-minimum-pressure

> In 9 awake implanted rats, the minimum bladder pressure of unstimulated voiding cycles was 10.3 mmHg (95% CI 7.4 to 13.2).

In [ ]:
t1("table1-minimum-pressure")

**Result: reproduced.** Mean minimum pressure 11.74 mmHg (95% CI 8.47 to 15.01, 9 rats), within
the printed CI 7.4 to 13.2 under every reading; it is 1.44 mmHg above the printed 10.3, and equals the
11.741 (8.473 to 15.009) of the dataset's own R notebook output. Recomputed from the traces (`/Voids`
onsets, rat means), the lowest pressure before the next void gives 6.26 mmHg (pre-stimulation
cycles), outside the CI, and the lowest within 60 s of the peak gives 8.00, inside it; the
derivative's definition is not stated.

### table1-threshold-pressure

> The relative threshold pressure of unstimulated voiding was 6.6 mmHg (95% CI 4.7 to 8.4), n = 9 rats.

In [ ]:
t1("table1-threshold-pressure")

**Result: reproduced.** Mean relative threshold pressure 6.96 mmHg (95% CI 5.15 to 8.77, 9
rats), within the printed CI 4.7 to 8.4 under every reading. Recomputed from the traces, threshold
at the onset gives 8.64 mmHg against the lowest pressure before the next void (outside the CI)
and 6.90 against the lowest within 60 s of the peak (inside); threshold at the start of the
HFPOs, the proxy here for urine release, gives 16.07 and 13.98 mmHg, outside under both.

### table1-peak-pressure

> The relative peak (closing) pressure of unstimulated voiding was 51.2 mmHg (95% CI 39.5 to 63.0), n = 9 rats.

In [ ]:
t1("table1-peak-pressure")

**Result: reproduced.** Mean relative peak pressure 46.59 mmHg (95% CI 33.79 to 59.39, 9 rats),
within the printed CI 39.5 to 63.0 under every reading; 4.6 mmHg below the printed 51.2. Recomputed from the traces it is 46.33
(or 44.59) mmHg, inside the CI under every reading.

### table1-contraction-duration

> The contraction duration of unstimulated voiding was 28.9 s (95% CI 17.1 to 37.5), n = 9 rats.

In [ ]:
t1("table1-contraction-duration")

**Result: reproduced.** Mean contraction duration 32.19 s (95% CI 16.04 to 48.33, 9 rats) with
18-273's two complete cycles, and 26.06 s (16.96 to 35.17, 8 rats) with that rat left out as
the dataset's R code does (its output gives 26.064, n = 8); 30.30 s over 26 pooled cycles. All lie within the printed CI 17.1
to 37.5. One cycle of 18-273 lasts 134.2 s. Recomputed from the traces, the time
above threshold (reading b) gives 32.08 s, inside the CI; onset to the return within 10% of
the minimum (reading a, with that margin a choice made here) gives 48.05 s, outside it.

### table1-intervoid-interval

> The inter-void interval of unstimulated voiding was 7.6 min (95% CI 5.9 to 9.4), n = 9 rats.

In [ ]:
t1("table1-intervoid-interval")

**Result: reproduced.** Mean inter-void interval 6.66 min (95% CI 5.74 to 7.59, 9 rats), within
the printed CI 5.9 to 9.4 under every reading; 0.94 min below the printed 7.6. Recomputed from the traces (onset to onset
within one wave) it is 6.66 min (7 rats with an interval before stimulation) or 6.23 min (first
three marked voids, 9 rats), inside the CI.

### table1-hfpo-frequency

> High-frequency pressure oscillations during voiding contractions had a centre frequency of 10.7 Hz (95% CI 8.3 to 13.0), n = 9 rats.

In [ ]:
t1("table1-hfpo-frequency")

**Result: reproduced.** Mean HFPO frequency 9.82 Hz (95% CI 7.21 to 12.43), within the printed
CI 8.3 to 13.0, but from 7 rats: the derivative has no HFPO value for 12-081 and 12-083, where
the paper states n = 9. Recomputed from the traces, the spectrum
peak gives 8.05 Hz (9 rats) and the count of oscillation peaks 7.11 Hz (7 rats), both below the
CI; these use a 3 to 30 Hz band-pass and an HFPO period defined here, so the derivative's
value decides the outcome. The outcome rests on an assumption: that the derivative's HFPO
frequency, whose method neither the paper nor the dataset states, matches one of the criterion's
readings. Nothing recomputed here supports it, and it comes from 7 rats.

### table1-comparable-to-published

> Voiding parameters after array implantation were comparable to published values in naive
> rats, including Wiedmann et al. (2020) for threshold pressure, peak pressure, contraction
> duration and inter-void interval.

The comparison values are Wiedmann et al. (2020) as printed in the paper's Table 1 (an outside
input, quoted in the claim's criterion). Reproduced when each of the four 95% CIs computed here
overlaps the printed Wiedmann CI.

In [ ]:
W20 = {"mmhg_init_rel": (1.8, 7.3), "mmhg_peak_rel": (15.3, 74.0), "contraction_dur_s": (25.1, 49.3), "ivi_minute": (5.3, 10.6)}
rows = []
for pid, (lo, hi) in W20.items():
    for rd, x in (("a1", ratm[pid]), ("a2", ratm_drop[pid])):
        m, l, h, n = P.mean_ci(x)
        rows.append(dict(parameter=pid, reading=rd, mean=m, ci_low=l, ci_high=h, wiedmann_ci=(lo, hi),
                         overlaps=(l <= hi) and (h >= lo)))
cm = pd.DataFrame(rows)
print(cm.round(2).to_string())
print("overlapping, reading a1:", int(cm[cm.reading == "a1"].overlaps.sum()), "of 4; reading a2:",
      int(cm[cm.reading == "a2"].overlaps.sum()), "of 4")

**Result: reproduced.** The 95% CIs computed here overlap the Wiedmann et al. (2020) CIs
printed in Table 1 for all four parameters under both readings: relative threshold 5.15 to
8.77 against 1.8 to 7.3 mmHg, relative peak 33.79 to 59.39 against 15.3 to 74.0 mmHg, contraction
duration 16.04 to 48.33 (or 16.96 to 35.17) against 25.1 to 49.3 s, and inter-void interval
5.74 to 7.59 against 5.3 to 10.6 min.

## Stimulation responses in awake rats (blocked)

stim-evoked-voiding, stim-void-trials, stim-early-void, stim-nonvoiding,
stim-voiding-to-nonvoiding and fig3e-eight-week-responses need, for each stimulation train in
eight rats (and the two 8-week rats), whether urine was released. The files hold stimulation
trains for four rats only (18-196, 18-243, 18-262, 18-294) and no annotation of urine release;
the 8-week rats have no stimulation trains. These claims are blocked as `data_missing`.

What the four rats allow is shown below as partial tests, which cannot decide the claims (their
counts are of eight rats). The deposited schema of the cystometry files says what `/Voids`
holds:

In [ ]:
txt = P.read_bytes("files/docs/hdf5_cystometry_schema.json").decode()
print(txt[txt.index('Group "/Voids"'):txt.index('Group "/Stimuli"')])

The schema lists the `/Voids` columns with units only; it does not say whether an entry means
that urine was released, or how an entry was chosen. Each `/Voids` entry is read here as a
voiding contraction marked by the authors (an assumption: the group is named "Voids", the
derivative's cycles come from it, and its entries have the threshold, peak and post-void
baseline of a voiding contraction). Per train: the marked void whose onset (`ThresholdTime`)
follows the train onset most closely, timed from the onset and from the end of the train (the
claims file's two starts of the 30 s window); the time since the previous marked void, against
the rat's mean interval between marked voids before the first train (stim-early-void readings:
under half, under a quarter); and the pressure rise after the train (highest 1 s mean pressure
from the onset to 30 s after the end, minus the mean of the 30 s before the onset), against the
rat's relative threshold pressure (stim-nonvoiding reading (a): a rise above it with no marked
void within 30 s of the end).

In [ ]:
rows = []
for s in cy_meta[cy_meta.stim_trains > 0]["sub"]:
    W, V, S = CY[s]
    V = V.sort_values("ThresholdTime").reset_index(drop=True)
    times = S["Time"].values if "Time" in S else np.array([S.attrs["Time"]])
    D = float(S.attrs["StimDuration"])
    pre_v = V[V.ThresholdTime < times.min()]
    ivi_pre = np.array([P.ivi_min(W, V, k) for k in pre_v.index])
    ivi_pre = ivi_pre[np.isfinite(ivi_pre)]
    ivi_src = "marked voids before the first train"
    if len(ivi_pre) == 0 and "R" + s in set(dv.subject_id):
        ivi_pre = dv[(dv.subject_id == "R" + s) & (dv.parameter_id == "ivi_minute")].value.dropna().values
        ivi_src = "derivative table"
    rel_thr = np.nanmean([P.void_params(W, V, k, next_onset=V.ThresholdTime[k + 1] if k + 1 < len(V) else np.inf).get("thr_a_rel", np.nan)
                          for k in pre_v.index])
    for t0 in times:
        dt = V.ThresholdTime.values - t0
        nxt = dt[dt > -1]
        prev = V.ThresholdTime.values[V.ThresholdTime.values < t0]
        w = P._wave_at(W, t0)
        rise = np.nan
        if w is not None and w["t"][0] <= t0 - 30 and t0 + D + 30 <= w["t"][-1]:
            ps = P.smooth(w["p"], float(w["attrs"]["SampleRate"]))
            tt = w["t"]
            rise = float(np.nanmax(ps[(tt >= t0) & (tt <= t0 + D + 30)]) - np.nanmean(w["p"][(tt >= t0 - 30) & (tt < t0)]))
        rows.append(dict(sub=s, onset_s=round(t0, 1), mA=S["Stim"].values[list(times).index(t0)] if "Stim" in S else S.attrs.get("Stim"),
                         next_void_from_onset_s=round(nxt.min(), 1) if len(nxt) else np.nan,
                         next_void_from_end_s=round(nxt.min() - D, 1) if len(nxt) else np.nan,
                         since_prev_void_min=(t0 - prev.max()) / 60 if len(prev) else np.nan,
                         rat_mean_ivi_pre_min=ivi_pre.mean() if len(ivi_pre) else np.nan, n_ivi_pre=len(ivi_pre), ivi_from=ivi_src,
                         rise_mmHg=rise, rat_rel_thr_mmHg=rel_thr))
st = pd.DataFrame(rows)
st["void_30s_onset"] = st.next_void_from_onset_s <= 30
st["void_30s_end"] = st.next_void_from_end_s <= 30
st["early_half"] = st.void_30s_end & (st.since_prev_void_min < 0.5 * st.rat_mean_ivi_pre_min)
st["early_quarter"] = st.void_30s_end & (st.since_prev_void_min < 0.25 * st.rat_mean_ivi_pre_min)
st["nonvoid_a"] = (st.rise_mmHg > st.rat_rel_thr_mmHg) & ~st.void_30s_end
print(st.round(2).to_string())
per = st.groupby("sub")[["void_30s_onset", "void_30s_end", "early_half", "early_quarter", "nonvoid_a"]].sum()
per["trains"] = st.groupby("sub").size()
print(per.to_string())
print("rats (of the 4 with trains) with a marked void within 30 s of onset:", int((per.void_30s_onset > 0).sum()),
      "; of the end:", int((per.void_30s_end > 0).sum()),
      "; trains followed by a marked void within 30 s of the end:", int(st.void_30s_end.sum()), "of", len(st))
print("rats with such voids in 3 to 6 trains:", int(per.void_30s_end.between(3, 6).sum()))
print("rats with an early evoked void, under half / a quarter of the mean interval:",
      int((per.early_half > 0).sum()), "/", int((per.early_quarter > 0).sum()))
print("rats with a non-voiding response under reading (a):", int((per.nonvoid_a > 0).sum()), "of 4")

fig3e-eight-week-responses, the half on void parameters. The 8-week rats (15-759, 15-760)
have marked voids but no trains, so only their filling-evoked voids can be compared: their
mean parameters over the first three marked voids, recomputed from the traces as for Table 1,
against the range of the per-rat means of the nine short-term rats (pre-stimulation cycles),
widened by 10% of each bound, under the (a) definitions and under the (b) ones.

In [ ]:
rows = []
for s in ["15-759", "15-760"]:
    W, V, S = CY[s]
    V = V.sort_values("ThresholdTime").reset_index(drop=True)
    for k in range(min(3, len(V))):
        d = P.void_params(W, V, k, next_onset=V.ThresholdTime[k + 1] if k + 1 < len(V) else np.inf)
        d.update(sub=s, void=k, ivi_min=P.ivi_min(W, V, k))
        rows.append(d)
LT = pd.DataFrame(rows)
short = TR[TR.selection == "pre-stimulation"].groupby("sub").mean(numeric_only=True)
rows = []
for defs in (("a", ["minimum", "thr_a_rel", "peak_rel", "dur_a", "ivi_min", "hfpo_a"]),
             ("b", ["minimum", "thr_b_rel", "peak_rel", "dur_b", "ivi_min", "hfpo_b"])):
    for s in ["15-759", "15-760"]:
        m = LT[LT["sub"] == s].mean(numeric_only=True)
        inside = []
        for par in defs[1]:
            lo, hi = short[par].min(), short[par].max()
            lo_w, hi_w = lo - 0.1 * abs(lo), hi + 0.1 * abs(hi)
            ok = bool(np.isfinite(m.get(par, np.nan)) and lo_w <= m[par] <= hi_w)
            inside.append(ok)
            rows.append(dict(definitions=defs[0], sub=s, parameter=par, value=m.get(par, np.nan), short_min=lo, short_max=hi, inside_widened=ok))
        print(f"definitions {defs[0]}, {s}: {sum(inside)} of 6 parameters inside the widened short-term range")
print(pd.DataFrame(rows).round(2).to_string())

**Partial results (the claims stay blocked).** Reading each `/Voids` entry as a marked void,
all four rats with trains have a marked void starting within 30 s of a train's onset and of its
end (4 of 9 trains), where stim-evoked-voiding counts 7 of 8 rats; no rat has such voids after
3 to 6 trains, as only 18-294 has more than two trains. No train is followed by a void earlier
than half the rat's mean interval (0 rats under either stim-early-void reading). Under reading
(a) of stim-nonvoiding no train in the four rats gives a pressure rise above the rat's relative
threshold without a marked void (0 of 4 rats; the rises without a void are 1.2 to 7.8 mmHg). For
fig3e, the 8-week rats' mean parameters lie inside the widened short-term range for 5 (15-759)
and 6 (15-760) of six parameters under the (a) definitions, and 3 and 4 under the (b) ones, where
neither 8-week rat has an HFPO period under the proxy used here. None of these decides a claim,
as the counts are of eight stimulated rats and urine release is not recorded.

## Histology, c-Fos and post-mortem (blocked)

fig4-atf3-rare, fig4ef-axons-intact, fig5-cfos-distribution, fig5-cfos-no-side-difference,
fig5-no-lamina1 and macroscopic-no-pathology need images, c-Fos counts or post-mortem notes. None
is in the dataset (data check), although the README lists "Summary derived data from c-Fos
activation mapping". They are blocked as `data_missing`.

## Summary

Summary. A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| anes-ecap-recorded | reproduced (assumed peak detection) |
| anes-ecap-latencies | not reproduced (assumed peak detection) |
| anes-conduction-velocity | not reproduced (assumed peak detection) |
| anes-threshold-order | not reproduced (assumed peak detection) |
| fig2c-p3-above-p1 | reproduced under one reading (assumed peak detection, unadjusted Dunn) |
| fig2ab-one-rat-p2-below-p1 | not reproduced (assumed peak detection) |
| awake-ecap-latencies | not reproduced (assumed peak detection) |
| awake-threshold-increase | blocked, metadata_insufficient |
| impedance-increase | partially reproduced (8-week value not in the data) |
| electrode-failures | blocked, data_missing |
| colonic-pressure | not reproduced |
| no-fecal-expulsion-or-erection | blocked, data_missing |
| table1-minimum-pressure | reproduced |
| table1-threshold-pressure | reproduced |
| table1-peak-pressure | reproduced |
| table1-contraction-duration | reproduced |
| table1-intervoid-interval | reproduced |
| table1-hfpo-frequency | reproduced (7 rats; assumed: the derivative's unstated method) |
| table1-comparable-to-published | reproduced (with the Wiedmann values printed in Table 1) |
| stim-evoked-voiding, stim-void-trials, stim-early-void, stim-nonvoiding, stim-voiding-to-nonvoiding, fig3e-eight-week-responses | blocked, data_missing |
| fig4-atf3-rare, fig4ef-axons-intact, fig5-cfos-distribution, fig5-cfos-no-side-difference, fig5-no-lamina1, macroscopic-no-pathology | blocked, data_missing |

How the analysis differs from the paper's. The paper does not say how ECAP peaks were
identified, so P1 to P3 come from rules assumed here: the first three positive peaks above k
noise SDs after the artefact at the highest current (main rule), the three largest, the triple
inside the paper's own ranges, and the first three of the inverted traces. The latency,
velocity, threshold-order, one-rat and awake outcomes are the same under all four; the
fig2c pass holds under each rule in some reading, unadjusted. The acute files hold currents up
to 1.6 mA in logarithmic steps, not 0 to 2 mA in 0.1 mA steps, and no 0 mA trace; the lowest
current stands for 0 mA. The 0.05 uV threshold of the Methods is applied as written although
noise alone passes it in most files; raised pass levels are shown as variants. Table 1 is
tested on the derivative table the authors deposited, and recomputed from the pressure traces
as a check, with definitions chosen here; peak pressure, the time above threshold and the
inter-void interval agree with Table 1 under that recomputation, while minimum pressure and
threshold pressure at onset depend on how the minimum is taken, threshold pressure at the start
of the HFPOs is above the CI under both, and the HFPO frequency comes out lower. The
stimulation claims are blocked; partial counts over the four rats with trains are shown. The
colonic windows are the claims file's.

What was found about the dataset. The dataset holds the acute and awake ECAPs, the colonic
recordings, the unstimulated cystometry with a derivative table for Table 1, and impedances; it
holds none of the histology, c-Fos counts or observation notes (the README lists c-Fos summary
data that are not there), and no annotation of urine release. The ECAP files are at 200 kHz with
a 500 to 3000 Hz filter, not the stated 100 kHz and 300 to 5000 Hz. The awake ECAP files are
seven subjects (four at day 0, three at day 7) with no link between weeks. The derivative
cystometry table differs from the `/Voids` tables of the same rats in most cycles and does not
give Table 1's exact means, though each lies within Table 1's CI; it has HFPO values for 7 of 9
rats. In 15-758 the colonic response precedes each recorded stimulus time by the length of a
train. The per-array impedance file reproduces the paper's values exactly; the per-electrode
files cover 8 arrays. The dataset lists 29 subjects; the paper reports 24 rats.

In [ ]:
print("runtime %.1f min; peak memory %.2f GB" % ((time.time() - T0) / 60,
      resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
print("files read:", len(P.READ))